In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/one-million-clicks-later/sample_submission.csv
/kaggle/input/one-million-clicks-later/train.csv
/kaggle/input/one-million-clicks-later/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from scipy.stats import randint, uniform
import joblib


In [3]:
train = pd.read_csv("/kaggle/input/one-million-clicks-later/train.csv")
test = pd.read_csv("/kaggle/input/one-million-clicks-later/test.csv")


In [4]:

train.columns = [c.strip() for c in train.columns]
test.columns = [c.strip() for c in test.columns]


to_num = ['liked', 'commented', 'subscribed_after', 'recommended', 'clicked']
for col in to_num:
    if col in train.columns:
        train[col] = pd.to_numeric(train[col], errors='coerce').fillna(0).astype(int)
    if col in test.columns:
        test[col] = pd.to_numeric(test[col], errors='coerce').fillna(0).astype(int)


train = train.fillna(0)
test = test.fillna(0)


In [5]:

train['watch_ratio'] = (train['watch_time'] / (train['video_duration'] + 1)).clip(0, 1)
test['watch_ratio'] = (test['watch_time'] / (test['video_duration'] + 1)).clip(0, 1)


train['timestamp_parsed'] = pd.to_datetime(train['timestamp'], format='%d-%m-%Y %H:%M', errors='coerce')
test['timestamp_parsed'] = pd.to_datetime(test['timestamp'], format='%d-%m-%Y %H:%M', errors='coerce')

train['hour'] = train['timestamp_parsed'].dt.hour.fillna(0).astype(int)
test['hour'] = test['timestamp_parsed'].dt.hour.fillna(0).astype(int)

train['dayofweek'] = train['timestamp_parsed'].dt.dayofweek.fillna(0).astype(int)
test['dayofweek'] = test['timestamp_parsed'].dt.dayofweek.fillna(0).astype(int)


In [6]:

for col in ['user_id', 'video_id']:
    if col in train.columns:
        freq = train[col].value_counts(dropna=False)
        train[f'{col}_freq'] = train[col].map(freq).fillna(0).astype(int)
        test[f'{col}_freq'] = test[col].map(freq).fillna(0).astype(int)


In [7]:

from sklearn.preprocessing import LabelEncoder

cat_cols = [c for c in ['category','device','watch_time_of_day','source'] if c in train.columns]
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
    le.fit(combined)
    train[col+'_enc'] = le.transform(train[col].astype(str))
    test[col+'_enc'] = le.transform(test[col].astype(str))


In [8]:

features = [
    'video_duration', 'watch_time', 'watch_ratio', 'hour', 'dayofweek',
    'liked', 'commented', 'subscribed_after', 'recommended'
]


for c in cat_cols:
    features.append(c + '_enc')


for c in ['user_id_freq', 'video_id_freq']:
    if c in train.columns:
        features.append(c)


features = [f for f in features if f in train.columns and f in test.columns]

X = train[features].copy()
y = train['clicked'].copy()
X_test = test[features].copy()


In [9]:
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)


In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import numpy as np


X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

val_probs = rf.predict_proba(X_val)[:, 1]
val_probs = np.nan_to_num(val_probs, nan=0.0)

threshs = np.linspace(0.1, 0.9, 41)
best_thresh = 0.5
best_f1 = 0

for t in threshs:
    preds = (val_probs >= t).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1:
        best_f1 = score
        best_thresh = t

print("Best threshold:", best_thresh)
print("Best F1 on validation:", best_f1)


Best threshold: 0.33999999999999997
Best F1 on validation: 0.25525436387487194


In [26]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

common_cols = [c for c in X.columns if c in test_fixed.columns]
test_fixed = test_fixed[common_cols]

test_scaled = pd.DataFrame(scaler.transform(test_fixed), columns=common_cols)

test_probs = rf.predict_proba(test_scaled)[:, 1]
test_probs = np.nan_to_num(test_probs, nan=0.0)

submission = pd.DataFrame({'id': test['id'], 'clicked': (test_probs >= best_thresh).astype(int)})
submission.to_csv('submission.csv', index=False)
submission.head()



,id,clicked
0,53363,0
1,293669,0
2,52195,0
3,260007,0
4,602213,0
